# AGender - Step 0: download the datasets and check them (CPU)

- **LAT** (Kuang et al. 2022): downloaded from PLOS (supporting file S1, 435 KB).
- **Copenhagen Networks Study**: calls, SMS, Facebook friendships and genders (about 0.5 MB). Figshare refuses
  downloads from Kaggle machines, so attach the Kaggle dataset `copenhagen-networks-study` (an unchanged copy;
  every file is checked against the MD5 sum figshare publishes). Without it, the notebook tries figshare.

Nothing is changed in the data. The notebook counts and describes: missing cells, rows where every session count is
zero, values that repeat too often, columns that are copies of each other, age and gender distributions.

**Kaggle settings:** Accelerator = **None** (CPU). Internet = **On**.
**Inputs to add:** `agender-code` and `copenhagen-networks-study`.
**Safe for "Save & Run All":** each part is one step; an error is recorded and the next step runs;
`step0_report.txt` is always written. **Expected run time:** 1-3 minutes.

The files are kept in the output folder (`data/`), so this notebook's output can be attached to the
next steps as the dataset `agender-step0`.

In [ ]:
import os
import sys
from pathlib import Path

# The AGender code is attached as a Kaggle dataset. Find the `agender` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('audit.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender').exists():
        os.symlink(package.parent, code_root / 'agender')

sys.path.insert(0, str(code_root))
import agender
print('AGender code loaded from', Path(agender.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_outputs/step0')
TIME_BUDGET_MINUTES = 60

In [ ]:
import json

from agender import audit, data, environment
from agender.runner import StepRunner

DATA = OUTPUT / 'data'
DATA.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}
downloads = []


def save_summary():
    summary['steps'] = runner.steps
    summary['downloads'] = downloads
    with open(OUTPUT / 'step0_summary.json', 'w') as f:
        json.dump(summary, f, indent=1, default=str)


summary['environment'] = runner.run('measure the machine', environment.collect)
save_summary()

In [ ]:
# ---- LAT ----
record = runner.run('download LAT from PLOS', data.fetch_lat, DATA)
if record:
    downloads.append(record)
    print(record)
lat = runner.run('read LAT', data.read_lat, DATA / 'lat_s1.zip')
if lat is not None:
    print(lat.shape)
    display(lat.head())
    summary['lat'] = runner.run('check LAT', audit.audit_lat, lat)
save_summary()

In [ ]:
# ---- Copenhagen Networks Study ----
found = data.find_copenhagen(INPUT)
print('Copenhagen files attached:', {k: str(v) for k, v in found.items()} or 'none')
if found:
    records = runner.run('copy Copenhagen (MD5 checked)', data.copy_copenhagen, found, DATA)
else:
    records = runner.run('download Copenhagen from figshare', data.fetch_copenhagen, DATA)
if records:
    downloads.extend(records)
    for r in records:
        print(r)
copenhagen = runner.run('read Copenhagen', data.read_copenhagen, DATA)
if copenhagen:
    summary['copenhagen'] = runner.run('check Copenhagen', audit.audit_copenhagen, copenhagen)
save_summary()

In [ ]:
# ---- Report (always written) ----
save_summary()
print(audit.write_report(summary, OUTPUT / 'step0_report.txt'))
print('Failed or skipped steps:', runner.failed() or 'none')
print('Outputs in', OUTPUT, ':', sorted(str(p.relative_to(OUTPUT)) for p in OUTPUT.rglob('*') if p.is_file()))